# S5 dev-full 베이스라인 검색

KURE-v1 + 512/64 고정 청킹(D-04) 전체 인덱스를 전수 검색한 dev-full 베이스라인 점수와 세부 분석을 기록한다. SPEC 미결 2(수치 목표)를 정하는 근거 자료다.

- 질의 임베딩은 dev 질의만 내보내 Colab T4에서 만들었다(D-07, `docs/slices/05-retrieve.md` 실행 기록). 검색과 채점은 로컬에서 했다
- 순위 파일은 `rag.search index.scope=full`, 채점은 `rag.eval.score --layer dev-full`의 결과다. 이 노트북은 그 결과를 다시 계산해 확인하고 나눠 본다
- 질의 원문은 예시 몇 건만 출력한다(D-07)

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import gold_ranks, load_gold, metric_table, min_docs, per_query, read_run
from rag.index import load_cfg, load_embeddings
from rag.search import first_docs, rank_with_pool

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg(["index.scope=full"])
EV = cfg.eval
INDEX_DIR = ROOT / cfg.paths.index_dir
RUN = ROOT / cfg.paths.runs_dir / f"{INDEX_DIR.parent.name}-dev-full.csv"
SCORE = RUN.with_suffix(".json")
RUN_SMALL = ROOT / cfg.paths.runs_dir / f"{INDEX_DIR.parent.name}-dev-small.csv"
N_EXAMPLES = 3  # 질의 원문을 출력하는 실패 사례 수

## 1. 데이터

In [3]:
gold = load_gold(cfg, "dev-full")
print(gold.shape, "회의", gold["conference_number"].nunique())
gold.head(3)

(3016, 4) 회의 752


,doc_id,qna_type,conference_number,meeting_name
qid,,,,
020437-0001,020437-dbb52cdb01,추출형,020437,특별위원회
020473-0001,020473-1ff4bc2f51,추출형,020473,본회의
020483-0001,020483-9ff0983fb4,추출형,020483,본회의


In [4]:
pd.concat({"질의 수": gold["meeting_name"].value_counts(),
           "비율": gold["meeting_name"].value_counts(normalize=True).round(3)}, axis=1)

,질의 수,비율
meeting_name,,
국정감사,1225,0.406
소위원회,1214,0.403
특별위원회,307,0.102
예산결산특별위원회,158,0.052
본회의,112,0.037


In [5]:
pd.concat({"질의 수": gold["qna_type"].value_counts(),
           "비율": gold["qna_type"].value_counts(normalize=True).round(3)}, axis=1)

,질의 수,비율
qna_type,,
추출형,2727,0.904
단답형,289,0.096


In [6]:
meta = json.loads((INDEX_DIR / "meta.json").read_text(encoding="utf-8"))
{k: meta[k] for k in ["n_docs", "n_chunks", "dim", "device", "dtype"]}

{'n_docs': 38516,
 'n_chunks': 118039,
 'dim': 1024,
 'device': 'cuda',
 'dtype': 'float16'}

## 2. 질의 임베딩

In [7]:
cached = np.load(ROOT / cfg.paths.query_emb)
q_emb = cached["emb"]
norms = np.linalg.norm(q_emb, axis=1)
print(q_emb.shape, q_emb.dtype)
print("순서가 채점 순서와 같음:", cached["qids"].tolist() == gold.index.tolist())
print("노름 오차 최대:", float(np.abs(norms - 1).max()), "NaN:", bool(np.isnan(q_emb).any()))

(3016, 1024) float32
순서가 채점 순서와 같음: True
노름 오차 최대: 1.1920928955078125e-07 NaN: False


## 3. 검색

In [8]:
with (INDEX_DIR / "chunks.jsonl").open(encoding="utf-8") as f:
    chunks = pd.DataFrame([json.loads(line) for line in f])
chunk_doc_ids = chunks["doc_id"].to_numpy()
chunk_emb = load_embeddings(INDEX_DIR)
print(chunks.shape, chunk_emb.shape, f"{chunk_emb.nbytes / 2**20:.0f} MiB")

(118039, 6) (118039, 1024) 461 MiB


질의 하나의 상위 5개 문서. 정답 문서에 표시한다.

In [9]:
qid = gold.index[0]
q_text = json.loads(next(line for line in (ROOT / cfg.paths.queries_dev).open(encoding="utf-8")))
print(q_text["qid"] == qid, q_text["query"])
scores = q_emb[0] @ chunk_emb.T
docs, _ = rank_with_pool(scores, chunk_doc_ids, 5, cfg.retriever.chunk_pool)
best = pd.Series(scores).groupby(chunk_doc_ids).max()
pd.DataFrame({"doc_id": docs, "최고 청크 점수": best[docs].round(4).to_numpy(),
              "정답": [d == gold.loc[qid, "doc_id"] for d in docs]}, index=range(1, 6))

True 공공근로사업의 장애인 고용 비율이 미흡한데, 공공근로사업에 대해 장애인 고용의 의무 비율을 높이는 노력을 해주실 것인가요?


,doc_id,최고 청크 점수,정답
1,020437-dbb52cdb01,0.7144,True
2,050933-2a2efa239e,0.6627,False
3,051403-e62a6ca27a,0.6476,False
4,050933-438011bf14,0.6470,False
5,049576-f940d038ea,0.6442,False


`chunk_pool`(200) 확인. 모든 질의에서 상위 200청크만 정렬한 결과가 전체 정렬 결과와 같은지, 전체 정렬로 넘어간 질의가 몇 건인지 본다.

In [10]:
r = cfg.retriever
same, fallbacks, rows = 0, 0, []
for start in range(0, len(gold), r.query_batch):
    block = q_emb[start : start + r.query_batch] @ chunk_emb.T
    for qid_, row in zip(gold.index[start : start + r.query_batch], block, strict=True):
        docs, full = rank_with_pool(row, chunk_doc_ids, r.top_k, r.chunk_pool)
        fallbacks += full
        same += docs == first_docs(np.argsort(-row, kind="stable"), chunk_doc_ids, r.top_k)
        rows += [{"qid": qid_, "rank": r, "doc_id": d} for r, d in enumerate(docs, 1)]
print(f"전체 정렬과 같음 {same}/{len(gold)}, 전체 정렬로 넘어감 {fallbacks}")

전체 정렬과 같음 3016/3016, 전체 정렬로 넘어감 0


In [11]:
run = read_run(RUN)
print(run.shape, "순위 파일과 같음:", pd.DataFrame(rows).equals(run))

(30160, 3) 순위 파일과 같음: True


## 4. 평가

In [12]:
ranks = gold_ranks(run, gold, min_docs(EV.recall_ks, EV.mrr_k))
scores_q = per_query(ranks, EV.recall_ks, EV.mrr_k)
table = metric_table(scores_q, gold, EV.breakdowns)
table.round(4)

,group,value,n,recall@1,recall@5,recall@10,mrr@10
0,all,all,3016,0.7062,0.8584,0.8909,0.7707
1,meeting_name,국정감사,1225,0.7298,0.8824,0.9102,0.7952
2,meeting_name,본회의,112,0.5714,0.7768,0.8125,0.6553
3,meeting_name,소위원회,1214,0.7290,0.8674,0.8954,0.7864
4,meeting_name,예산결산특별위원회,158,0.6266,0.8038,0.8671,0.7023
5,meeting_name,특별위원회,307,0.6124,0.7850,0.8371,0.6880
6,qna_type,단답형,289,0.6990,0.8408,0.8651,0.7570
7,qna_type,추출형,2727,0.7070,0.8603,0.8937,0.7721


In [13]:
saved = pd.DataFrame(json.loads(SCORE.read_text(encoding="utf-8"))["metrics"])
print("채점 결과 파일과 같음:", np.allclose(saved[table.columns[3:]], table[table.columns[3:]]))

채점 결과 파일과 같음: True


정답 순위 분포. 10위 밖은 상위 10개 문서에 정답이 없는 질의다.

In [14]:
bins = pd.cut(ranks.fillna(11), [0, 1, 5, 10, 11], labels=["1", "2~5", "6~10", "10위 밖"])
pd.concat({"질의 수": bins.value_counts(sort=False),
           "누적 비율": bins.value_counts(sort=False).cumsum().div(len(bins)).round(4)}, axis=1)

,질의 수,누적 비율
rank,,
1,2130,0.7062
2~5,459,0.8584
6~10,98,0.8909
10위 밖,329,1.0000


Recall@5 회의 단위 bootstrap 95% 구간(SPEC 판정과 같은 반복·seed). 이후 실험의 차이를 이 구간 폭과 견줘 볼 수 있다.

In [15]:
b = EV.bootstrap
point, lo, hi = paired_bootstrap(scores_q["recall@5"], gold["conference_number"],
                                 b.n_resamples, b.seed, b.alpha)
print(f"Recall@5 {point:.4f} [{lo:.4f}, {hi:.4f}], 폭 {100 * (hi - lo):.2f}%p")

Recall@5 0.8584 [0.8428, 0.8741], 폭 3.13%p


dev-small 점수(같은 인덱스 설정, 방해 문서 약 1,000개). SPEC에 따라 dev-full과 비교하지 않고 따로 적는다.

In [16]:
gold_small = load_gold(cfg, "dev-small")
ranks_small = gold_ranks(read_run(RUN_SMALL), gold_small, min_docs(EV.recall_ks, EV.mrr_k))
metric_table(per_query(ranks_small, EV.recall_ks, EV.mrr_k), gold_small, []).round(4)

,group,value,n,recall@1,recall@5,recall@10,mrr@10
0,all,all,1039,0.8739,0.9673,0.9817,0.9151


## 5. 실패 사례

정답이 10위 밖인 질의에서 1위 문서가 정답과 같은 회의(`conference_number`)에서 나왔는지 본다. 같은 회의의 비슷한 발언이 방해 문서가 되는지 가늠한다.

In [17]:
def conf(doc_ids):
    # doc_id 앞부분이 회의 번호다(예: 020437-dbb52cdb01)
    return np.array([d.split("-")[0] for d in doc_ids])


top1 = run[run["rank"] == 1].set_index("qid")["doc_id"]
miss = ranks.index[ranks.isna()]
fail = pd.DataFrame({"gold": gold.loc[miss, "doc_id"], "top1": top1[miss]})
fail["같은 회의"] = conf(fail["gold"]) == conf(fail["top1"])
hit1 = ranks.index[ranks == 1]
print("10위 밖", len(miss), "그중 1위가 같은 회의:", round(fail["같은 회의"].mean(), 3))

10위 밖 329 그중 1위가 같은 회의: 0.064


비교: 정답이 1위가 아닌 모든 질의에서 1위 문서가 같은 회의인 비율.

In [18]:
wrong1 = ranks.index[ranks != 1]
same_conf = conf(gold.loc[wrong1, "doc_id"]) == conf(top1[wrong1])
print("1위 오답", len(wrong1), "그중 같은 회의:", round(same_conf.mean(), 3))

1위 오답 886 그중 같은 회의: 0.111


10위 밖 질의 몇 건의 질의, 정답 문서·1위 문서 앞부분(질의 원문은 `N_EXAMPLES`건만).

In [19]:
queries_dev = {}
with (ROOT / cfg.paths.queries_dev).open(encoding="utf-8") as f:
    for line in f:
        q = json.loads(line)
        queries_dev[q["qid"]] = q["query"]
first_chunk = chunks.drop_duplicates("doc_id").set_index("doc_id")["text"]
for qid_ in fail.sample(N_EXAMPLES, random_state=cfg.seed).index:
    g, t = fail.loc[qid_, "gold"], fail.loc[qid_, "top1"]
    print(f"[{qid_}] {queries_dev[qid_]}")
    print(f"  정답 {g} ({gold.loc[qid_, 'meeting_name']}): {first_chunk[g][:120]!r}")
    print(f"  1위 {t} (같은 회의 {fail.loc[qid_, '같은 회의']}): {first_chunk[t][:120]!r}\n")

[047529-0016] 국민이 어린 아동을 상대로 한 성폭력 및 강력범죄에 대해 어떤 감정을 가지고 있는지는 무엇인가요?
  정답 047529-76b1425571 (예산결산특별위원회): '국회입법조사처에 의뢰하면 이 내용이 있습니다. 이 내용은 2015년에 저희가 받아서 만든 내용입니다. 지금은 또 달라졌을 것입니다.   많은 법조인이 태완이법을 만들 때도 그렇게 얘기했습니다, ‘서영교 의원! 법조인'
  1위 047211-d871eebb3f (같은 회의 False): '늦게까지 고생 많으십니다.  변호사회에서 이번에 세 번째 대법관 추천을 받아서 이른바 대법관후보자 삼수생이다 하는데, 맞습니까? 세 번째인 것은 맞습니다. 언제, 언제 추천을 받으셨습니까? 2011년, 2016년 그'

[048468-0013] 정부가 세금을 많이 거두어들인 것에 대한 문제 인식 부재에 대해 어떤 대응을 취할 계획이 있나요?
  정답 048468-8f1fd1bc45 (국정감사): '대구 달성군 출신 추경호 위원입니다.  국세청장님, 평소에 경제계 소상공인 자영업자 등 현장 목소리 듣고 계십니까? 예, 그렇습니다. 그분들 요새 경제 괜찮다고 그럽니까, 경제 어렵다고 그럽니까? 많은 어려움이 있다'
  1위 048128-8df4f83eca (같은 회의 False): '대전 서구을 출신의 박범계 위원입니다.  지금 존경하는 곽대훈 위원님이 질의한 내용에 덧붙여서 좀 여쭤보겠습니다.  어제 고위당정회의가 있었는데요, 제가 이해하기로는 이렇게 이해를 했습니다. 그러니까 장하성 정책실장'

[051405-0009] 감사보고서에는 어떤 내용이 포함되어 있으며, 주요 내용 중 어떤 부분이 감사 결과를 지지하는 데 중요한 역할을 했나요?
  정답 051405-4ab06c95fa (특별위원회): '오늘 청문회 과정을 지켜보면서 최재형 전 감사원장에 대한 국정감사를 하는 것인지 최재해 감사원장후보자에 대한 인사청문회를 실시하는 것인지를 제가 헷갈릴 정도인데……    (자료를 들어 보이며)  후보

## 6. 정리

In [20]:
summary = table[table["group"] == "all"][["recall@1", "recall@5", "recall@10", "mrr@10"]]
summary.assign(**{"R@5 95% 하한": lo, "R@5 95% 상한": hi}).round(4)

,recall@1,recall@5,recall@10,mrr@10,R@5 95% 하한,R@5 95% 상한
0,0.7062,0.8584,0.8909,0.7707,0.8428,0.8741


- 수치 목표(SPEC 미결 2)는 이 노트북을 보고 사용자가 정한다. 결정은 `docs/DECISIONS.md`에 기록한다
- SPEC 판정 규칙의 채택 기준은 기준 실험 대비 +1.0%p 이상이고 구간 하한이 0보다 큰 것이다